In [ ]:
"""PyCharm (No LangChain) — OpenRouter chat loop + token guard (compact, our style)"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import tiktoken  # นับโทเคนแบบคร่าว ๆ
from dotenv import load_dotenv
from openai import OpenAI

# ============================================================================
# ENVIRONMENT SETUP (fail-fast)
# ============================================================================
load_dotenv()
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")
if not OPENROUTER_API_KEY:  raise RuntimeError("Missing OPENROUTER_API_KEY in .env")
if not OPENROUTER_BASE_URL: raise RuntimeError("Missing OPENROUTER_BASE_URL in .env")

# ============================================================================
# SIMPLE TOKEN COUNTER (approx, for GPT-4o family use o200k_base)
# ============================================================================
ENCODING_NAME = "o200k_base"
enc = tiktoken.get_encoding(ENCODING_NAME)

def count_tokens_messages(messages: list[dict]) -> int:
    # นับแบบง่าย: รวม role + content แล้ว encode
    total = 0
    for m in messages:
        total += len(enc.encode(m.get("role",""))) + len(enc.encode(m.get("content","")))
    return total

# ============================================================================
# MAIN
# ============================================================================
def main():
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

    # ตั้งค่าเริ่มต้น
    system_msg = {"role": "system", "content": "You are a helpful assistant. Topic: pets and salons."}
    conversation = [system_msg]

    # งบโทเคนแบบเรียบง่าย (สำหรับ GPT-4o context กว้างมาก แต่กำหนดเผื่อไว้)
    MAX_RESPONSE_TOKENS = 250
    TOKEN_LIMIT = 12_000  # กำหนดเพดานคร่าว ๆ เพื่อกันล้น

    print("I am a helpful assistant. I can talk about pets and salons. What would you like to talk about?")

    try:
        while True:
            user_text = input("> ").strip()
            if not user_text:
                continue

            conversation.append({"role": "user", "content": user_text})

            # ถ้าโทเคนใกล้ลิมิต ให้ตัดบทสนทนาต้น ๆ ออก (เว้น system)
            while count_tokens_messages(conversation) + MAX_RESPONSE_TOKENS >= TOKEN_LIMIT and len(conversation) > 2:
                del conversation[1]  # ลบข้อความเก่าที่สุดหลัง system

            resp = client.chat.completions.create(
                model="openai/gpt-4o",
                messages=conversation,
                temperature=0.8,
                max_tokens=MAX_RESPONSE_TOKENS,
                top_p=0.95,
                frequency_penalty=0,
                presence_penalty=0,
                # user="amit",  # ตัวเลือก: ติดแท็กผู้ใช้
            )
            reply = resp.choices[0].message.content.strip()
            conversation.append({"role": "assistant", "content": reply})
            print("\n" + reply)
            # ตัวเลือก: แสดงการใช้โทเคนถ้ามี
            usage = getattr(resp, "usage", None)
            if usage and getattr(usage, "total_tokens", None) is not None:
                print(f"(Tokens used: {usage.total_tokens})\n")
            else:
                print()
    except KeyboardInterrupt:
        print("\nBye!")

# ============================================================================
# SCRIPT EXECUTION
# ============================================================================
if __name__ == "__main__":
    main()


I am a helpful assistant. I can talk about pets and salons. What would you like to talk about?

Hello! How can I assist you today? Are you looking for information about pets, salons, or something else?
(Tokens used: 47)


Nice to meet you, Phonxay! How can I assist you today? Are you interested in learning more about pet care, grooming, or perhaps something else?
(Tokens used: 96)


Your name is Phonxay. How can I assist you further?
(Tokens used: 124)


In the realm of AI and language models, a "context window" refers to the amount of text that the model can consider or "remember" at one time while generating responses. For example, when you chat with an AI like me, I use the context window to keep track of the ongoing conversation. This helps maintain the relevance and coherence of responses based on previous exchanges.

For practical purposes, this means there is a limit to how much prior conversation or information can be used to generate a response. If a conversation or text excee